# E7: fine-tune Whisper-small on selected VoxPopuli data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e7_finetune_colab.ipynb)

Experiment E7 of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md)
asks whether the choice of training data matters when the amount does not change. Four strategies
each chose 5 hours of the E6 pool of Italian VoxPopuli, with three seeds; the selections are in
[`experiments/E7/results/selections.json`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E7/results/selections.json)
and the hyperparameters, fixed in advance, in
[`experiments/E7/finetune.toml`](https://github.com/ivannamor/multilingual-asr-lab/blob/main/experiments/E7/finetune.toml).

For each selection, this notebook fine-tunes Whisper-small with LoRA, merges and converts the model
to CTranslate2, and transcribes the VoxPopuli test split (in-domain) and the FLEURS Italian test
split (out-of-domain) with the same decoding as the base model, which is transcribed too.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*. Set `SESSION_SEEDS` below: all
   models take about 3 to 4 hours, so the seeds can be split between sessions, e.g. `[0]` (with the
   base model, about 1.5 hours) and then `[1, 2]`. Then *Runtime → Run all* and keep the tab open.
2. At the end the browser downloads `e7_runs_seeds-….zip` (if it does not, download it from the
   *Files* panel): the predictions and `run.json` of every transcription and the training record
   of every model, no audio and no weights.
3. In a local clone, from the repository root: `python3 -m zipfile -e e7_runs_seeds-….zip .`

If the session is interrupted, run all cells again: finished models are skipped and a transcription
resumes where it stopped.

## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where the experiment scripts expect them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    # The training extra adds PEFT; Colab already has PyTorch and Transformers
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[training]"], check=True)
print(Path.cwd())

## 2. Settings

In [ ]:
SESSION_SEEDS = [0, 1, 2]  # the seeds trained in this session; the base model goes with seed 0
SPEC_PATH = Path("experiments/E7/finetune.toml")  # strategies, hyperparameters, test sets
SELECTIONS_PATH = Path("experiments/E7/results/selections.json")
MODELS_DIR = Path("models/e7")  # deleted after each model's transcriptions
RECORDS_DIR = Path("experiments/E7/results/training")  # one training record per model

### GPU setup

Same setup as the quickstart notebook: install the CUDA 12 libraries that CTranslate2 is built for
and load them explicitly. Unlike the quickstart, this notebook needs the GPU and stops if it is not
usable.

In [ ]:
import ctypes

import ctranslate2
import numpy as np

if ctranslate2.get_cuda_device_count() == 0:
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

In [ ]:
import nvidia.cublas
import nvidia.cudnn
from faster_whisper import WhisperModel

pending = [
    path
    for package in (nvidia.cublas, nvidia.cudnn)
    for path in sorted(Path(next(iter(package.__path__))).glob("lib/*.so*"))
]
for _ in range(5):  # the libraries depend on each other: retry the ones that failed
    failed = []
    for path in pending:
        try:
            ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(path)
    pending = failed
    if not pending:
        break

check = WhisperModel("tiny", device="cuda", compute_type="float16")
list(check.transcribe(np.zeros(16_000, dtype=np.float32), language="it")[0])
print("GPU ready")

## 3. Data

The E6 pool (about 2.4 GB) and the VoxPopuli test split (about 0.9 GB), from the Hugging Face Hub at
the pinned revision, and the FLEURS Italian test split, exactly as the local scripts download them.

In [ ]:
import tomllib

from asr_lab.data.ingest import ingest_fleurs
from asr_lab.data.voxpopuli import ingest_voxpopuli, load_data_spec

spec = tomllib.loads(SPEC_PATH.read_text(encoding="utf-8"))
for data_spec in (spec["selection"]["quality_spec"], spec["evaluation"]["test_spec"]):
    data = load_data_spec(Path(data_spec))
    if not data.manifest.exists():
        ingest_voxpopuli(data)
    print(data.manifest)
test_sets = {name: Path(path) for name, path in spec["evaluation"]["test_sets"].items()}
if not test_sets["fleurs"].exists():
    ingest_fleurs("it", "test", test_sets["fleurs"].parent)
print(test_sets)

## 4. Fine-tuning and transcription

For each model: train (about 5 minutes), convert, transcribe both test sets (about 10 minutes),
then delete the weights. The loss is printed every 10 steps.

In [ ]:
import json
import shutil

import torch

from asr_lab.inference.faster_whisper import FasterWhisperModel
from asr_lab.inference.run import transcribe_manifest
from asr_lab.training.lora import TRAINING_FILE, fine_tune, load_training_config
from asr_lab.training.selection import selected_entries

config = load_training_config(SPEC_PATH)
decoding = {
    "beam_size": spec["evaluation"]["beam_size"],
    "hallucination_silence_threshold": spec["evaluation"]["hallucination_silence_threshold"],
}
models = [("base", 0)] if 0 in SESSION_SEEDS else []
models += [(s, seed) for seed in SESSION_SEEDS for s in spec["selection"]["strategies"]]


def run_dir(name: str, test_set: str) -> Path:
    return Path(f"experiments/runs/e7-{name}-{test_set}")


def finished(path: Path) -> bool:
    run_file = path / "run.json"
    return run_file.exists() and json.loads(run_file.read_text())["finished_at"] is not None


for strategy, seed in models:
    name = "base" if strategy == "base" else f"{strategy.replace('_', '-')}-s{seed}"
    if all(finished(run_dir(name, test_set)) for test_set in test_sets):
        print(f"{name}: done")
        continue
    print(f"=== {name}")
    if strategy == "base":
        manifest, entries = Path(), []
    else:
        manifest, entries = selected_entries(SELECTIONS_PATH, strategy, seed)
    model_dir = MODELS_DIR / name
    ct2_dir = fine_tune(
        config,
        entries,
        manifest,
        model_dir,
        seed=seed,
        description={"strategy": strategy, "selections": SELECTIONS_PATH.as_posix()},
    )
    RECORDS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(model_dir / TRAINING_FILE, RECORDS_DIR / f"{name}.json")
    torch.cuda.empty_cache()  # the training's memory, before CTranslate2 loads the model
    model = FasterWhisperModel(str(ct2_dir), device="cuda", **decoding)
    for test_set, test_manifest in test_sets.items():
        transcribe_manifest(model, test_manifest, run_dir(name, test_set))
    del model
    shutil.rmtree(model_dir)

## 5. Download the results

In [ ]:
import zipfile

archive = Path(f"e7_runs_seeds-{'-'.join(map(str, SESSION_SEEDS))}.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for strategy, seed in models:
        name = "base" if strategy == "base" else f"{strategy.replace('_', '-')}-s{seed}"
        zipped.write(RECORDS_DIR / f"{name}.json", (RECORDS_DIR / f"{name}.json").as_posix())
        for test_set in test_sets:
            for file in ("predictions.jsonl", "run.json"):
                path = run_dir(name, test_set) / file
                zipped.write(path, path.as_posix())
                if file == "run.json":
                    run = json.loads(path.read_text())
                    print(f"{path.parent.name}: RTF {run['rtf'] or float('nan'):.3f}")
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass